# Cribl decrypt — Sentinel notebook

Query the at-rest **encrypted** Cribl logs in Log Analytics and view the **decrypted** values on demand.
Runs Python server-side (in Azure ML compute or locally) — **no CORS, no trusted-host, no browser gates**.

**How it decrypts:** it calls the deployed Azure Function, which pulls the AES key from Key Vault and returns plaintext.
Key material never touches this notebook.


## 1. Install dependencies (first run only)


In [ ]:
%pip install -q azure-monitor-query azure-identity pandas requests


## 2. Configuration
_The function endpoint includes a function key — treat this notebook as a secret, or move the key to a prompt/Key Vault for shared use._


In [ ]:
WORKSPACE_ID    = "<WORKSPACE_ID>"
FUNCTION_ENDPOINT = "https://cribldec-func-ukz4eyn7t4iik.azurewebsites.net/api/decrypt?code=<FUNCTION_KEY>"
TABLE           = "CriblEncrypted_CL"
LOOKBACK_DAYS   = 1


## 3. Query the encrypted logs
Authenticates with your Azure identity (az login locally, or the compute's managed identity in Azure ML).


In [ ]:
from datetime import timedelta
import pandas as pd
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient

client = LogsQueryClient(DefaultAzureCredential())
query = f"""{TABLE}
| where TimeGenerated > ago({LOOKBACK_DAYS}d)
| project TimeGenerated, User_s, Action_s, SourceHost_s, EncryptedField_s"""

resp = client.query_workspace(WORKSPACE_ID, query, timespan=timedelta(days=LOOKBACK_DAYS))
df = pd.DataFrame(data=resp.tables[0].rows, columns=resp.tables[0].columns)
print(f'{len(df)} rows')
df.head(20)


## 4. Decrypt the encrypted column
Sends the distinct tokens to the Function (server-side HTTP — no browser involved) and joins plaintext back to each row.


In [ ]:
import requests

tokens = [t for t in df['EncryptedField_s'].dropna().unique().tolist() if t]
resp = requests.post(FUNCTION_ENDPOINT, json={"tokens": " ".join(tokens)}, timeout=30)
resp.raise_for_status()
results = resp.json()['results']
lookup = {r['input']: (r['plaintext'] if r.get('ok') else f"<{r.get('error','failed')}>") for r in results}

df['Decrypted'] = df['EncryptedField_s'].map(lookup)
df[['TimeGenerated','User_s','Action_s','EncryptedField_s','Decrypted']]


## 5. (Optional) Decrypt locally from Key Vault instead of the Function
If you'd rather not call the Function, pull the key from Key Vault and AES-decrypt in-notebook.
Requires your identity to have **Key Vault Secrets User** on the vault. The crypto is the same ~20 lines as the Function.


In [ ]:
# import base64, json, re
# from azure.keyvault.secrets import SecretClient
# from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
# VAULT_URL = 'https://cribldeckvukz4eyn7t4iik.vault.azure.net/'
# sc = SecretClient(VAULT_URL, DefaultAzureCredential())
# def b64(s):
#     s=s.strip(); s+='='*((4-len(s)%4)%4); return base64.b64decode(s)
# def strip_pkcs7(d):
#     p=d[-1]; return d[:-p] if 1<=p<=16 and d[-p:]==bytes([p])*p else d
# def decrypt(tok):
#     m=re.match(r'^#?([A-Za-z0-9]+):([A-Za-z0-9+/=]*):([A-Za-z0-9+/=]+)#?$', tok.strip())
#     kid, ivb, ct = m.group(1), m.group(2), m.group(3)
#     meta=json.loads(sc.get_secret(f'cribl-key-{kid}').value)
#     key=bytes.fromhex(meta['keyHex']); iv=b64(ivb) if ivb else b'\x00'*16
#     dec=Cipher(algorithms.AES(key), modes.CBC(iv)).decryptor()
#     return strip_pkcs7(dec.update(b64(ct))+dec.finalize()).decode()
# df['Decrypted'] = df['EncryptedField_s'].map(decrypt)
# df[['User_s','Action_s','EncryptedField_s','Decrypted']]
